In [1]:
# Housekeeping

import os
import pandas as pd
import dateutil
import re
import shutil 
import numpy as np
import importlib
import utils  
importlib.reload(utils)
from utils import * 

In [29]:
genotype = "D1.1"
start_date = "11-01-2021"
end_date = "12-05-2025"
locations = "Antarctica,North America,South America"
date_range = start_date + "--" + end_date
start_date_gisaid = "2021-11-01"
end_date_gisaid = "2025-12-05"

home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"
os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")

andersen_metadata = home + "Andersen/avian-influenza/metadata/"
ncbi_virus_metadata = home + "NCBI_Virus/downloads/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/" # "_Antarctica_North_America_South_America/"
gisaid_metadata = home + "GISAID/downloads/" + start_date_gisaid + "--" + end_date_gisaid + "_" + locations.replace(",", "_").replace(" ", "_") + "/"

andersen_ncbi_virus_gisaid = home + "Combinations/NCBI_Virus_Andersen_GISAID/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/"



In [35]:
# Get sequences

os.chdir(home)

fasta = pd.DataFrame()
for dirpath, dirs, files in os.walk(andersen_ncbi_virus_gisaid):
    if len(files) > 0: # If we have any files that need to be moved
        for file in files:
            file_name = os.path.join(dirpath, file)
            if genotype in file_name and "PB2" in file_name:
                print(file_name)
                fasta = fasta_df_complete(file_name, states_ref)
                print(fasta)
                break
    break

C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/NCBI_Virus_Andersen_GISAID/11-01-2021--12-05-2025_Antarctica_North_America_South_America/all_D1.1_PB2_2021-11-01--2025-12-05.fasta
                                                 Header  \
0     PX661684|A/chicken/OR/W250070001-1/2025|H5N1|U...   
1     PX661692|A/chicken/OR/W250070001-2/2025|H5N1|U...   
2     SRR35586416|A/chicken/SD/25-024635-001-origina...   
3     SRR35586404|A/Turkey/MT/25-024658-001-original...   
4     SRR35949426|A/Domestic_Grower/Meat-type_Turkey...   
...                                                 ...   
3000  EPI_ISL_19756050|A/great_horned_owl/USA/003295...   
3001  EPI_ISL_19593640|A/goose/USA/24-032297-001/202...   
3002  EPI_ISL_20249109|A/Whooping_crane/SK/FAV-0436-...   
3003  EPI_ISL_20249110|A/Whooping_crane/SK/FAV-0436-...   
3004  EPI_ISL_19726293|A/Nevada/10/2025|H5N1|USA-NV|...   

                  Isolate_Id  \
0               W250070001-1   
1               W250070001-2 

In [36]:
# NCBI Virus Metadata

os.chdir(ncbi_virus_metadata)
ncbi_virus_metadata_csv = pd.read_csv("sequences.csv")
ncbi_virus_metadata_csv["Identifier"] = ncbi_virus_metadata_csv["Accession"].apply(lambda x: x.split(".")[0])

# print(ncbi_virus_metadata_csv)

ncbi_virus_people = ncbi_virus_metadata_csv[["Submitters", "Organization", "Identifier"]]

print(ncbi_virus_people)

fasta_ncbi_virus = fasta.merge(ncbi_virus_people, on="Identifier", how="left")

print(fasta_ncbi_virus)

                                               Submitters  \
0       Rivetti,A.V. Jr., Reischak,D., Carnegie,L., Ot...   
1       Rivetti,A.V. Jr., Reischak,D., Carnegie,L., Ot...   
2       Rivetti,A.V. Jr., Reischak,D., Carnegie,L., Ot...   
3       Rivetti,A.V. Jr., Reischak,D., Carnegie,L., Ot...   
4       Rivetti,A.V. Jr., Reischak,D., Carnegie,L., Ot...   
...                                                   ...   
118139  Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...   
118140  Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...   
118141  Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...   
118142  Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...   
118143  Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...   

                                             Organization Identifier  
0       Ministerio da Agricultura e Pecuaria, Laborato...   PV659823  
1       Ministerio da Agricultura e Pecuaria, Laborato...   PV659824  
2       Ministerio da Agricultura e Pecuaria, Laborato

In [37]:
# Andersen Metadata

os.chdir(andersen_metadata)
andersen_metadata_csv = pd.read_csv("SraRunTable_automated_normalized.tsv", delimiter="\t")

print(andersen_metadata_csv)
andersen_people = andersen_metadata_csv[["Run", "Center Name"]]
andersen_people = andersen_people.rename(columns={"Run":"Identifier", "Center Name":"Organization"})

fasta_ncbi_virus_andersen = fasta_ncbi_virus.merge(andersen_people, on=["Identifier"], how="left", suffixes=(None, "_SRA"))
fasta_ncbi_virus_andersen["Organization"] = fasta_ncbi_virus_andersen["Organization"].combine_first(fasta_ncbi_virus_andersen["Organization_SRA"])
print(fasta_ncbi_virus_andersen)

               Run Assay Type  AvgSpotLen      Bases    BioProject  \
0      SRR28752446        WGS      146.11   93605195  PRJNA1102327   
1      SRR28752447        WGS      241.29   86080323  PRJNA1102327   
2      SRR28752448        WGS      250.30   75035343  PRJNA1102327   
3      SRR28752449        WGS      146.61   59363690  PRJNA1102327   
4      SRR28752450        WGS      251.31  119232569  PRJNA1102327   
...            ...        ...         ...        ...           ...   
13280  SRR36202286        WGS      147.13  118909569   PRJNA980729   
13281  SRR36202287        WGS      147.21  126097467   PRJNA980729   
13282  SRR36202288        WGS      146.33   89907532   PRJNA980729   
13283  SRR36202289        WGS      148.33   89276755   PRJNA980729   
13284  SRR36202290        WGS      148.14   79763616   PRJNA980729   

          BioSample BioSampleModel     Bytes Center Name Collection_Date  ...  \
0      SAMN41019184          Viral  30074178   USDA-NVSL      2024-03-16  ... 

In [44]:
# GISAID Metadata

os.chdir(gisaid_metadata)
gisaid_metadata_csv = pd.read_excel("gisaid_epiflu_isolates.xls")

print(gisaid_metadata_csv[["Isolate_Id", "Isolate_Submitter"]])
gisaid_people = gisaid_metadata_csv[["Isolate_Id", "Isolate_Submitter"]]

gisaid_people = gisaid_people.rename(columns={"Isolate_Id":"Identifier", "Isolate_Submitter":"Submitters"})
gisaid_people["Organization"] = gisaid_people["Submitters"].apply(lambda x: x.split("(")[1].split(")")[0] if x==x and "(" in x else x)
fasta_ncbi_virus_andersen_gisaid = fasta_ncbi_virus_andersen.merge(gisaid_people, on=["Identifier"], how="left", suffixes=(None, "_GISAID"))
fasta_ncbi_virus_andersen_gisaid["Submitters"] = fasta_ncbi_virus_andersen_gisaid["Submitters"].combine_first(fasta_ncbi_virus_andersen_gisaid["Submitters_GISAID"])
fasta_ncbi_virus_andersen_gisaid["Organization"] = fasta_ncbi_virus_andersen_gisaid["Organization"].combine_first(fasta_ncbi_virus_andersen_gisaid["Organization_GISAID"])

print(fasta_ncbi_virus_andersen_gisaid)

             Isolate_Id                                  Isolate_Submitter
0      EPI_ISL_20055781                                                NaN
1      EPI_ISL_19825740                                                NaN
2      EPI_ISL_19825728                                                NaN
3      EPI_ISL_19825727                                                NaN
4      EPI_ISL_19825726                                                NaN
...                 ...                                                ...
19681  EPI_ISL_19070288  Sandra Liliana Landazabal Castillo (National U...
19682  EPI_ISL_14777716                                                NaN
19683  EPI_ISL_19594058                                                NaN
19684  EPI_ISL_19594056                                                NaN
19685  EPI_ISL_19594053                                                NaN

[19686 rows x 2 columns]
                                                 Header  \
0     PX661684|

In [ ]:
os.chdir(home)

fasta_ncbi_virus_andersen_gisaid.to_csv("D1.1_PB2_metadata_credits.csv")